# Sample Corpus SQL Integrations — `DocumentStore`

Deep dive into **every SQL surface** of the medical + salvage sample corpus:

| Section | Focus |
|---------|--------|
| 0 | Setup + open DB |
| 1 | DDL / schema version |
| 2 | Claims table CRUD |
| 3 | Documents upsert + filters |
| 4 | `document_fields` ground truth |
| 5 | Joins: claim ↔ documents ↔ fields |
| 6 | Splits, source_kind, synthetic flags |
| 7 | Provenance events |
| 8 | Raw SQL analytics |
| 9 | Import / export round-trips |
| 10 | Page table hook (optional assets) |

The store uses stdlib `sqlite3` (same pattern as Discord `notes_store.py`) — no ORM required.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import sqlite3
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

CWD = Path.cwd().resolve()
# Walk up from docs/notebooks/ (or notebooks/) until pyproject.toml is found.
REPO_ROOT = next(
    (p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").exists()),
    None,
)
assert REPO_ROOT is not None, f"Could not find repo root from {CWD}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.docie import DociePipeline
from src.docie.applications import list_applications, load_application
from src.docie.eval import evaluate_application
from src.docie.pipeline import run_file
from src.storage import DocumentStore
from src.storage.sample_generator import generate_claim_bundle, generate_corpus
from src.storage.schema import DDL, SCHEMA_VERSION
from src.storage.training import (
    fit_tfidf_random_forest,
    prepare_both_applications,
)
from src.storage.types import ClaimRecord, DocumentRecord, FieldRecord
from src.utils.config import Config
from src.utils.io import load_jsonl, read_json, write_json

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
cfg = Config.load()

DEMO = REPO_ROOT / "data" / "notebook_demo" / "sample_corpus"
DEMO.mkdir(parents=True, exist_ok=True)
DB_PATH = DEMO / "documents.db"
EXPORTS = DEMO / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)
PREPARED = DEMO / "prepared"
MODELS = DEMO / "models"
MODELS.mkdir(parents=True, exist_ok=True)

SEED = 42
print(f"repo:     {REPO_ROOT}")
print(f"demo db:  {DB_PATH}")
print(f"exports:  {EXPORTS}")
print(f"schema v: {SCHEMA_VERSION}")
print(f"apps:     {list_applications()}")

In [ ]:
# Fresh demo DB for this notebook
if DB_PATH.exists():
    DB_PATH.unlink()
store = DocumentStore(DB_PATH)
corpus = generate_corpus(
    seed=SEED,
    medical_per_type=5,
    salvage_per_type=5,
    bundles_per_app=1,
    include_canonical_fixtures=True,
)
store.bulk_upsert(corpus.documents, claims=corpus.claims)
print(store.summary())

## 1. DDL / schema version

In [ ]:
print("SCHEMA_VERSION =", SCHEMA_VERSION)
print("--- DDL excerpt ---")
print("\n".join(DDL.strip().splitlines()[:40]))
print("...")

with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    indexes = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='index' ORDER BY name"
    ).fetchall()
print("tables:", [t[0] for t in tables])
print("indexes:", [i[0] for i in indexes])

## 2. Claims table CRUD

In [ ]:
claim = ClaimRecord(
    claim_id="CLM-DEMO-SQL-001",
    application="salvage_claims",
    carrier_name="American Family Insurance",
    state="WI",
    date_of_loss="2024-05-10",
    loss_type="collision",
    policy_number="AF-42-1002003",
    insured_name="Jamie Demo",
    metadata={"notebook": "sql_integrations"},
)
store.upsert_claim(claim)
loaded = store.get_claim("CLM-DEMO-SQL-001")
display(loaded.to_dict() if loaded else None)

# Update carrier branding via upsert
claim.carrier_name = "AmFam"
store.upsert_claim(claim)
print("updated carrier:", store.get_claim("CLM-DEMO-SQL-001").carrier_name)

## 3. Documents upsert + filter APIs

In [ ]:
doc = DocumentRecord(
    document_id="sal-demo-log-sql",
    claim_id="CLM-DEMO-SQL-001",
    application="salvage_claims",
    document_type="log",
    title="Demo Letter of Guarantee",
    text=(
        "LETTER OF GUARANTEE\n"
        "Heartland Bank Title Department\n"
        "Claim Number: CLM-DEMO-SQL-001\n"
        "VIN: 1HGCM82633A004352\n"
        "Year: 2018\n"
        "Make: Honda\n"
        "Model: Accord\n"
        "Payoff Amount: $3,100.00\n"
    ),
    source_kind="notebook_demo",
    is_synthetic=True,
    split="train",
    skeleton={
        "vehicle": {
            "vin": "1HGCM82633A004352",
            "year": "2018",
            "make": "Honda",
            "model": "Accord",
        }
    },
    fields=[
        FieldRecord("claim_id", "CLM-DEMO-SQL-001"),
        FieldRecord("vin", "1HGCM82633A004352"),
        FieldRecord("year", "2018"),
        FieldRecord("make", "Honda"),
        FieldRecord("model", "Accord"),
    ],
)
store.upsert_document(doc)

filters = [
    {"application": "salvage_claims"},
    {"application": "salvage_claims", "document_type": "log"},
    {"claim_id": "CLM-DEMO-SQL-001"},
    {"split": "train"},
    {"source_kind": "notebook_demo"},
]
for f in filters:
    n = len(store.list_documents(**f))
    print(f"{f} → {n}")

## 4. `document_fields` ground truth vs extracted roles

In [ ]:
store.set_fields(
    "sal-demo-log-sql",
    {"payoff_amount": "3100.00", "lienholder": "Heartland Bank"},
    role="annotation",
)
store.set_fields(
    "sal-demo-log-sql",
    {"vin": "1HGCM82633A004352", "make": "Honda"},
    role="extracted",
    confidence=0.91,
)

with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    field_rows = conn.execute(
        "SELECT field_name, field_value, field_role, confidence "
        "FROM document_fields WHERE document_id = ? "
        "ORDER BY field_role, field_name",
        ("sal-demo-log-sql",),
    ).fetchall()
pd.DataFrame([dict(r) for r in field_rows])

## 5. Joins: claim ↔ documents ↔ fields

In [ ]:
sql = """
SELECT
  c.claim_id,
  c.carrier_name,
  c.loss_type,
  d.document_id,
  d.document_type,
  d.split,
  COUNT(f.field_id) AS n_ground_truth_fields
FROM claims c
JOIN documents d ON d.claim_id = c.claim_id
LEFT JOIN document_fields f
  ON f.document_id = d.document_id AND f.field_role = 'ground_truth'
WHERE c.application = 'salvage_claims'
GROUP BY c.claim_id, d.document_id
ORDER BY c.claim_id, d.document_type
LIMIT 20
"""
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(sql, conn)
df

## 6. Splits, source kinds, synthetic flags

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    split_df = pd.read_sql_query(
        """
        SELECT application, document_type, split, COUNT(*) AS n
        FROM documents
        GROUP BY 1, 2, 3
        ORDER BY 1, 2, 3
        """,
        conn,
    )
    source_df = pd.read_sql_query(
        """
        SELECT source_kind, is_synthetic, COUNT(*) AS n
        FROM documents
        GROUP BY 1, 2
        """,
        conn,
    )
display(Markdown("### Split distribution"))
display(split_df)
display(Markdown("### Source kinds"))
display(source_df)

## 7. Provenance events

In [ ]:
store.add_provenance(
    stage="sql_notebook_demo",
    source="sample_corpus_sql_integrations",
    document_id="sal-demo-log-sql",
    claim_id="CLM-DEMO-SQL-001",
    detail={"action": "annotated_extra_fields"},
)
with sqlite3.connect(DB_PATH) as conn:
    prov = pd.read_sql_query(
        "SELECT event_id, document_id, claim_id, stage, source, detail_json "
        "FROM provenance_events",
        conn,
    )
prov

## 8. Raw SQL analytics

In [ ]:
analytics = """
SELECT
  application,
  document_type,
  ROUND(AVG(LENGTH(text)), 1) AS avg_chars,
  MIN(LENGTH(text)) AS min_chars,
  MAX(LENGTH(text)) AS max_chars,
  COUNT(*) AS n
FROM documents
GROUP BY application, document_type
ORDER BY application, document_type
"""
carrier_sql = """
SELECT carrier_name, COUNT(*) AS n_claims
FROM claims
GROUP BY carrier_name
ORDER BY n_claims DESC
"""
missing_vin = """
SELECT d.document_id, d.document_type, f.field_value AS vin
FROM documents d
LEFT JOIN document_fields f
  ON f.document_id = d.document_id
 AND f.field_name = 'vin'
 AND f.field_role = 'ground_truth'
WHERE d.application = 'salvage_claims'
ORDER BY CASE WHEN f.field_value IS NULL OR f.field_value = '' THEN 0 ELSE 1 END,
         d.document_id
LIMIT 15
"""
with sqlite3.connect(DB_PATH) as conn:
    display(Markdown("### Text length by type"))
    display(pd.read_sql_query(analytics, conn))
    display(Markdown("### Carrier mix (synthetic branding)"))
    display(pd.read_sql_query(carrier_sql, conn))
    display(Markdown("### Salvage VIN coverage"))
    display(pd.read_sql_query(missing_vin, conn))

## 9. Import / export round-trips

In [ ]:
roundtrip_path = EXPORTS / "sql_roundtrip_docie.jsonl"
exported = store.export_jsonl(roundtrip_path, format="docie", application="salvage_claims")
print("exported", exported)

# Import into a second DB to prove portability
db2 = DEMO / "documents_roundtrip.db"
if db2.exists():
    db2.unlink()
store2 = DocumentStore(db2)
imported = store2.import_docie_jsonl(roundtrip_path, source_kind="roundtrip")
print("imported", imported)
print(store2.summary())

# Field fidelity check
a = store.get_document("sal-log-001")
b = store2.get_document("sal-log-001")
print("field match:", a.ground_truth_fields() == b.ground_truth_fields() if a and b else None)

## 10. Optional `document_pages` hook

In [ ]:
# The pages table is reserved for rendered/OCR page assets (DICIE cache paths, etc.)
now = time.time()
with sqlite3.connect(DB_PATH) as conn:
    conn.execute(
        "INSERT INTO document_pages ("
        " document_id, page_index, image_path, width, height, dpi,"
        " ocr_text, words_json, created_at"
        ") VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)"
        " ON CONFLICT(document_id, page_index) DO UPDATE SET"
        " image_path=excluded.image_path, ocr_text=excluded.ocr_text",
        (
            "sal-demo-log-sql",
            0,
            str(DEMO / "pages" / "sal-demo-log-sql_p0.png"),
            1000,
            1200,
            200,
            "LETTER OF GUARANTEE ...",
            json.dumps([{"text": "LETTER", "bbox": [10, 10, 80, 30]}]),
            now,
        ),
    )
    conn.commit()
    pages = pd.read_sql_query(
        "SELECT document_id, page_index, width, height, dpi, image_path "
        "FROM document_pages",
        conn,
    )
pages

## Takeaways

- **Canonical house** for synthetic medical + salvage docs is SQLite via `DocumentStore`
- Ground-truth lives in `document_fields` with roles (`ground_truth` / `extracted` / `annotation`)
- Claim bundles are just multiple `documents` rows sharing `claim_id`
- JSONL import/export keeps DICIE + training pipelines file-compatible
- Raw SQL is available anytime for analytics that the Python API does not wrap